## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <cstdio>
#include <cstdlib>
#include <vector>
#include <algorithm>
#include <cassert>
using namespace std;

const int MAXN = 1050;

int n, A, B, lowbitLen;
int p[MAXN], pos[MAXN];
vector<int> answer;

inline int readInt() {
    int x = 0;
    char ch = getchar();
    while (ch < '0' || ch > '9') {
        if (ch == EOF) return 0;
        ch = getchar();
    }
    while (ch >= '0' && ch <= '9') {
        x = x * 10 + (ch - '0');
        ch = getchar();
    }
    return x;
}

void rebuildPos() {
    for (int i = 0; i < n; ++i) pos[p[i]] = i;
}

void useSwapMagic() {
    answer.push_back(0);
    for (int i = 0; i < n; ++i) {
        if (p[i] == A) p[i] = B;
        else if (p[i] == B) p[i] = A;
    }
    rebuildPos();
}

void useAddMagic(int v) {
    v %= n;
    if (v < 0) v += n;
    if (v == 0) return;
    answer.push_back(v);
    for (int i = 0; i < n; ++i) p[i] = (p[i] + v) % n;
    rebuildPos();
}

void useXorMagic(int v) {
    if (v == 0) return;
    answer.push_back(-v);
    for (int i = 0; i < n; ++i) p[i] ^= v;
    rebuildPos();
}

void getPairPos(int x, int y, int *px, int *py) {
    int delta = (y - x + n - lowbitLen + n) % n;
    *px = 0;
    *py = 0;
    for (int step = n / 2; step >= 2 * lowbitLen; step >>= 1) {
        if (delta >= step) {
            delta -= step;
            *py += step / 2;
        } else {
            *px += step / 2;
        }
    }
    *px += n / 2;
    *px += (x & (lowbitLen - 1));
    *py += (x & (lowbitLen - 1));
}

void applySwap(int x, int y) {
    if ((x / lowbitLen) % 2 == (y / lowbitLen) % 2) {
        int mid;
        if ((x / lowbitLen) % 2 == 0) mid = (x & (lowbitLen - 1)) + lowbitLen;
        else mid = (x & (lowbitLen - 1));
        applySwap(x, mid);
        applySwap(y, mid);
        applySwap(x, mid);
        return;
    }

    int pA, pB, pX, pY;
    getPairPos(A, B, &pA, &pB);
    getPairPos(x, y, &pX, &pY);

    useAddMagic((pX - x + n) % n);
    useXorMagic(pX ^ pA);
    useAddMagic((A - pA + n) % n);

    useSwapMagic();

    useAddMagic((pA - A + n) % n);
    useXorMagic(pX ^ pA);
    useAddMagic((x - pX + n) % n);
}

struct Permutation {
    int val[MAXN];
    int len;
    vector<int> ops;

    bool build() {
        bool vis[2005] = {false};
        for (int i = 0; i < len; ++i) {
            if (val[i] < 2005) {
                vis[val[i]] = true;
            } else {
                return false;
            }
        }
        for (int i = 0; i < len; ++i) {
            if (!vis[i]) return false;
        }

        if (len == 1) return true;

        Permutation leftPart, rightPart;
        leftPart.len = rightPart.len = len / 2;

        for (int i = 0; i < len / 2; ++i) {
            leftPart.val[i] = val[i * 2] / 2;
            rightPart.val[i] = val[i * 2 + 1] / 2;
        }

        if (!leftPart.build() || !rightPart.build()) {
            return false;
        }

        if (val[0] & 1) {
            ops.push_back((len == 2) ? 1 : -1);
        }

        int curXorL = 0;
        for (int x : leftPart.ops) {
            if (x > 0) {
                ops.push_back(-1);
                ops.push_back(1);
            } else {
                ops.push_back(x * 2);
                curXorL ^= (-x) * 2;
            }
        }
        if (curXorL) ops.push_back(-curXorL);

        int curXorR = 0;
        for (int x : rightPart.ops) {
            if (x > 0) {
                ops.push_back(1);
                ops.push_back(-1);
            } else {
                ops.push_back(x * 2);
                curXorR ^= (-x) * 2;
            }
        }

        if ((curXorR & (len / 2)) != (curXorL & (len / 2))) {
            return false;
        }

        if (curXorL >= len / 2) curXorL -= len / 2;
        if (curXorR >= len / 2) curXorR -= len / 2;
        if (curXorL != curXorR) return false;

        vector<int> merged;
        for (int x : ops) {
            if (merged.empty()) {
                merged.push_back(x);
            } else {
                if (x < 0 && merged.back() < 0) {
                    int last = merged.back();
                    merged.back() = -( (-last) ^ (-x) );
                    if (merged.back() == 0) {
                        merged.pop_back();
                    }
                } else {
                    merged.push_back(x);
                }
            }
        }
        ops = merged;
        return true;
    }
};

int main() {
    n = readInt();
    if (n == 0) return 0;
    A = readInt();
    B = readInt();

    for (int i = 0; i < n; ++i) p[i] = readInt();

    rebuildPos();

    lowbitLen = (A - B + n) % n;
    lowbitLen &= -lowbitLen;
    if (lowbitLen == 0) lowbitLen = n;

    if (lowbitLen > 1) {
        Permutation base;
        base.len = lowbitLen;
        for (int i = 0; i < n; ++i) {
            base.val[i] = p[i] & (lowbitLen - 1);
        }

        if (!base.build()) {
            printf("-1\n");
            return 0;
        }

        for (int x : base.ops) {
            if (x > 0) useAddMagic(x);
            else useXorMagic(-x);
        }
    }

    for (int rem = 0; rem < lowbitLen; ++rem) {
        vector<int> vec;
        for (int j = rem; j < n; j += lowbitLen) {
            vec.push_back(p[j]);
        }

        sort(vec.begin(), vec.end());

        bool ok = true;
        int ptr = 0;
        for (int j = rem; j < n; j += lowbitLen) {
            if (vec[ptr] != j) {
                ok = false;
                break;
            }
            ++ptr;
        }

        if (!ok) {
            printf("-1\n");
            return 0;
        }

        for (int j = rem; j < n; j += lowbitLen) {
            if (p[j] != j) {
                applySwap(j, p[j]);
            }
        }
    }

    for (int i = 0; i < n; ++i) {
        assert(p[i] == i);
    }

    printf("%d\n", (int)answer.size());
    for (int x : answer) {
        if (x == 0) {
            printf("0\n");
        } else if (x < 0) {
            printf("1 %d\n", -x);
        } else {
            printf("2 %d\n", x);
        }
    }

    return 0;
}

## B 长跑

In [ ]:
#include <iostream>
#include <cstdio>
#include <cstring>
#include <cmath>
#include <algorithm>
#include <queue>
using namespace std;

const int maxm = 2e4 + 5;
int n, l, ma, s;
struct Shop {
    int p, c;
} a[maxm];

struct Node {
    int now, s, id;
} st;

bool cmp(Shop a, Shop b) {
    if (a.p != b.p) return a.p < b.p;
    return a.c < b.c;
}

signed main() {
    while (scanf("%d%d%d%d", &n, &l, &ma, &s) != EOF) {
        for (int i = 1; i <= n; i++) {
            scanf("%d%d", &a[i].p, &a[i].c);
        }
        if (ma >= l) {
            puts("Yes");
            continue;
        }
        a[++n].p = l;
        a[n].c = 0;
        sort(a + 1, a + 1 + n, cmp);
        queue<Node> q;
        st = {0, s, 0};
        q.push(st);
        int ok = 0;
        while (!q.empty()) {
            Node x = q.front();
            q.pop();
            if (x.now == l) {
                ok = 1;
                break;
            }
            for (int i = x.id + 1; i <= n; i++) {
                if (a[i].p - x.now > ma) {
                    break;
                } else if (x.s >= a[i].c) {
                    Node t = {a[i].p, s - a[i].c, i};
                    q.push(t);
                }
            }
        }
        puts(ok ? "Yes" : "No");
    }
    return 0;
}

## C 最长回文

In [ ]:
#include<bits/stdc++.h>
using namespace std;
const int N = 3e5 + 10;
string a , b;
int pa[N] , pb[N] , res = 1;
string Manacher(string a , int *p)
{
    string t = "$#";
    for(auto i : a) t += i , t += '#';
    int mx = 0 , id = 0 ;
    int len = t.size() , ans = 0;
    for(int i = 1 ; i < len ; i ++)
    {
        p[i] = mx > i ? min(p[2 * id - i] , mx - i) : 1;
        while(t[i + p[i]] == t[i - p[i]]) p[i] ++ ;
        if(mx < i + p[i]) mx = i + p[i] , id = i;
        ans = max(ans , p[i] - 1);
    }
    res = max(res , ans);
    return t;
}
signed main()
{
    int n ;
    cin >> n >> a >> b;
    a = Manacher(a , pa) , b = Manacher(b , pb);
    n = n * 2 + 2;
    int ans = 1;
    for(int i = 2 ; i <= n ; i ++)
    {
        int len = max(pa[i] , pb[i - 2]);
        while(a[i - len] == b[i - 2 + len]) len ++;
        ans = max(ans , len - 1);
    }
    cout << ans << '\n';
    return 0;
}

## D 优惠券

In [ ]:
#include <iostream>
#include <set>
#include <cstdio>
#include <cstring>
using namespace std;

const int maxn = 1e5 + 100;
int num[maxn * 5];
int n;

int main() {
    char s[10];
    while (~scanf("%d", &n)) {
        set<int> Q;
        int ans;
        int ok = 0;
        set<int>::iterator it;
        for (int i = 0; i < n; i++) {
            scanf("%s", s);
            if (s[0] == '?') {
                Q.insert(i + 1);
            } else if (s[0] == 'I') {
                int tmp;
                scanf("%d", &tmp);
                if (ok)
                    continue;
                if (num[tmp] > 0) {
                    it = Q.lower_bound(num[tmp]);
                    if (it != Q.end()) {
                        Q.erase(it++);
                    } else {
                        if (ok == 0) {
                            ok = 1;
                            ans = i + 1;
                        }
                    }
                }
                num[tmp] = i + 1;
            } else {
                int tmp;
                scanf("%d", &tmp);
                if (ok)
                    continue;
                if (num[tmp] <= 0) {
                    it = Q.lower_bound(-num[tmp]);
                    if (it != Q.end()) {
                        Q.erase(it++);
                    } else {
                        if (ok == 0) {
                            ok = 1;
                            ans = i + 1;
                        }
                    }
                }
                num[tmp] = -(i + 1);
            }
        }

        if (ok)
            cout << ans << endl;
        else
            cout << -1 << endl;
    }
    return 0;
}

## E 任意点

In [ ]:
#include<bits/stdc++.h>
using namespace std;

int x[110],y[110];
int f[1100];
int n;
void init()
{
    for(int i=0; i<=1000; i++)
        f[i] = i;
}

int Find(int a)
{
    if(f[a] == a)
        return a;
    else
        return f[a] = Find(f[a]);
}

void unite(int a, int b)
{
    int fa = Find(a);
    int fb = Find(b);
    if(fa != fb)
        f[fa] = fb;
}

int main()
{
    while(~scanf("%d",&n))
    {
        init();
        for(int i=1; i<=n; i++)
            scanf("%d %d",&x[i],&y[i]);

        for(int i=1; i<=n; i++)
            for(int j=1; j<=n; j++)
                if(x[i] == x[j] || y[i] == y[j])
                    unite(i,j);
        int ans = 0;
        for(int i=1; i<=n; i++)
        {
            if(f[i] == i)
                ans++;
        }
        printf("%d\n",ans-1);
    }
    return 0;
}

## F 通配符匹配

In [ ]:
#include<iostream>
#include<cstdio>
#include<cstring>
#include<algorithm>
#define ll unsigned long long
ll key=19260817ll;
using namespace std;
ll pre[100010],mul[100010],h[20];int n,cnt,sp[20],stl[20],dp[15][100010];
char b[15][100010],a[100010],tmp[100010];
ll gethash(char s[],int len){
    ll re=0;int i;
    for(i=0;i<len;i++) re*=key,re+=(ll)s[i];
    return re;
}
int main(){
    scanf("%s",a);int i,j,k,l,len=strlen(a);ll t1;
    mul[0]=1;
    for(i=1;i<=100000;i++) mul[i]=mul[i-1]*key;
    if(a[0]=='*'||a[0]=='?') h[++cnt]=gethash(b[cnt],stl[cnt]=0);
    for(i=0;i<len;){
        if(a[i]=='*'||a[i]=='?'){sp[cnt]=(sp[cnt]||(a[i]=='*'));i++;}
        j=0;cnt++;
        while(a[i]!='*'&&a[i]!='?'&&i<len) b[cnt][j]=a[i],i++,j++;
        h[cnt]=gethash(b[cnt],stl[cnt]=j);
    }
    len++;
    if(a[len-2]=='*'||a[len-2]=='?') cnt++,sp[cnt]=stl[cnt]=h[cnt]=0;
    else sp[cnt]=0;
    scanf("%d",&n);
    for(l=1;l<=n;l++){
        memset(a,0,sizeof(a));
        scanf("%s",a);
        len=strlen(a);a[len]='$';len++;
        memset(dp,-1,sizeof(dp));dp[0][0]=2;pre[0]=0;
        for(i=0;i<len;i++) pre[i+1]=pre[i]*key+(ll)a[i];
        for(j=0;j<=len;j++){
            for(i=0;i<=cnt;i++){
                if(dp[i][j]==-1) continue;
                if(dp[i][j]==1) dp[i][j+1]=1;
                if(!dp[i][j]){
                    dp[i][j]=2;
                    if(dp[i][j+1]==-1) dp[i][j+1]=2;
                    if(dp[i][j+1]==0) dp[i][j+1]=3;
                    continue;
                }
                if(dp[i][j]==3){
                    dp[i][j]=2;
                    if(dp[i][j+1]==-1) dp[i][j+1]=2;
                    if(dp[i][j+1]==0) dp[i][j+1]=3;
                }
                t1=pre[j+stl[i+1]]-pre[j]*mul[stl[i+1]];
                if(t1==h[i+1]){
                    dp[i+1][j+stl[i+1]]=max(dp[i+1][j+stl[i+1]],sp[i+1]);
                }
            }
        }
        if(dp[cnt][len]!=-1) puts("YES");
        else puts("NO");
    }
}

## G 汉诺塔

In [ ]:
#include <iostream>
#include <cmath>
using namespace std;
typedef long long ll;

char a[4];
int s[9], p, n, i = 6;

ll f(int x) {
    if (x == 1) return (ll)2 * pow(3, n - 1) - 1;
    if (x) return (ll)pow(2, n) - 1;
    return (ll)pow(3, n - 1);
}

int main() {
    cin >> n;
    while (i--) {
        cin >> a;
        s[(a[0] - 'A') * 3 + a[1] - 'A'] = i;
    }
    if (s[1] > s[2]) {
        if (s[5] < s[3]) p = 1;
        else if (s[6] > s[7]) p = 2;
    } else if (s[7] < s[6]) p = 1;
    else if (s[3] > s[5]) p = 2;
    cout << f(p);
    return 0;
}

## H 马步距离

In [ ]:
#include <bits/stdc++.h>
using namespace std;
#define int long long

int x, y, s;

signed main() {
    int x1, y1, x2, y2;
    cin >> x1 >> y1 >> x2 >> y2;
    x = abs(x1 - x2), y = abs(y1 - y2);
    if (x == 2 && y == 2) s = 4;
    else if ((x == 1 && y == 0) || (x == 0 && y == 1)) s = 3;
    else s = max((x + y + 2) / 3, (max(x, y) + 1) / 2);
    if ((s - (x + y)) % 2) s++;
    cout << s;
    return 0;
}

## I 直方图最大矩形

In [ ]:
#include <bits/stdc++.h>
using namespace std;

class Solution {
public:
    int largestRectangleArea(vector<int>& heights) {
        int n = heights.size();
        stack<int> stk;
        int res = 0;
        for (int i = 0; i < n; i++) {
            while (!stk.empty() && heights[stk.top()] > heights[i]) {
                int curHeight = heights[stk.top()];
                stk.pop();
                int L = stk.empty() ? 0 : stk.top() + 1;
                res = max(res, (i - L) * curHeight);
            }
            stk.push(i);
        }

        while (!stk.empty()) {
            int curHeight = heights[stk.top()];
            stk.pop();
            int L = stk.empty() ? 0 : stk.top() + 1;
            res = max(res, (n - L) * curHeight);
        }

        return res;
    }
};

## J 消防局的设立

In [ ]:
#include <iostream>
#include <algorithm>
using namespace std;

const int maxN = 1010;
const int INF = 2000000000;

int N;
struct edge {
    int to;
    int next;
} sons[maxN];
int head[maxN] = {0};
int F[maxN][5];

int nowEdge = 0;
void addSon(int u, int v) {
    nowEdge++;
    sons[nowEdge].to = v;
    sons[nowEdge].next = head[u];
    head[u] = nowEdge;
}

void dfs(int now) {
    F[now][0] = 1;
    F[now][3] = 0;
    F[now][4] = 0;
    for (int i = head[now]; i; i = sons[i].next) {
        int s = sons[i].to;
        dfs(s);
        F[now][0] += F[s][4];
        F[now][3] += F[s][2];
        F[now][4] += F[s][3];
    }
    if (head[now] == 0) {
        F[now][1] = F[now][2] = 1;
    } else {
        F[now][1] = F[now][2] = INF;
        for (int i = head[now]; i; i = sons[i].next) {
            int s = sons[i].to;
            int F1 = F[s][0];
            int F2 = F[s][1];
            for (int j = head[now]; j; j = sons[j].next) {
                if (i == j) continue;
                int t = sons[j].to;
                F1 += F[t][3];
                F2 += F[t][2];
            }
            F[now][1] = min(F[now][1], F1);
            F[now][2] = min(F[now][2], F2);
        }
    }
    for (int i = 1; i <= 4; i++) {
        F[now][i] = min(F[now][i], F[now][i - 1]);
    }
}

int main() {
    cin >> N;
    for (int i = 2; i <= N; i++) {
        int f;
        cin >> f;
        addSon(f, i);
    }
    dfs(1);
    cout << F[1][2];
    return 0;
}